# 04 - Text to SQL

Turn a plain-English question into one SQLite query.

- **Prompt:** fixed rules in the system message; few-shot examples on a
  small made-up table; then the real schema text (notebook 02) and the
  question.
- **Output:** schema-enforced JSON with a short `plan`, `can_answer`
  and `sql`. Writing the plan first helps a small model pick the right
  columns and filters; `can_answer: false` lets it refuse questions the
  data cannot answer (no cost column -> no profit).
- **First accuracy check:** 16 questions with hand-written correct
  queries. Results are compared, not SQL text, so any query that returns
  the right numbers counts. No retries yet (notebook 05).

Becomes `src/helpers/sql_generator.py`.

In [1]:
# ============================================================
# 1. CONFIGURATION
# ============================================================

import json
import logging
import math
import sqlite3
import statistics
import time
from pathlib import Path

import ollama
import pandas as pd


BASE_DIR = (
    Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
)
DATASETS_DIR = BASE_DIR / "data" / "output_data" / "datasets"
MODEL_PROFILE_PATH = BASE_DIR / "data" / "output_data" / "model" / "model_profile.json"
OUTPUT_DIR = BASE_DIR / "data" / "output_data" / "text_to_sql"

OLLAMA_URL = "http://localhost:11434"
LLM_TEMPERATURE = 0
KEEP_ALIVE = "30m"

# Numbers count as equal within this relative tolerance (rounding).
RELATIVE_TOLERANCE = 0.005

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True,
)
logging.getLogger("httpx").setLevel(logging.WARNING)
logger = logging.getLogger("04_text_to_sql")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Model chosen by notebook 03 (no need to detect again).
with MODEL_PROFILE_PATH.open(encoding="utf-8") as file:
    model_settings = json.load(file)["selected"]

print("Model:", model_settings["model"], "| profile:", model_settings["profile"])

Model: qwen2.5:7b-instruct | profile: gpu


## 2. Helpers from earlier notebooks

Copied unchanged so this notebook runs on its own: the model call
(notebook 03) and the SQLite statistics functions (notebook 02). After
conversion they are imported from `src/helpers/`.

In [2]:
# ============================================================
# 2. HELPERS FROM EARLIER NOTEBOOKS
# ============================================================

# ---- notebook 03: model call --------------------------------

# Cap on generated tokens: a runaway answer becomes invalid JSON (and is
# retried) instead of running into the request timeout.
MAX_OUTPUT_TOKENS = 512


def get_client(timeout):
    return ollama.Client(host=OLLAMA_URL, timeout=timeout)


def chat_json(settings, messages, schema):
    """Call the model with a JSON schema; return (answer, stats)."""

    client = get_client(settings["request_timeout"])
    started = time.perf_counter()
    response = client.chat(
        model=settings["model"], messages=messages, format=schema,
        options={"temperature": LLM_TEMPERATURE,
                 "num_ctx": settings["num_ctx"],
                 "num_predict": MAX_OUTPUT_TOKENS},
        keep_alive=KEEP_ALIVE,
    )
    content = response["message"]["content"]

    try:
        answer = json.loads(content)
    except json.JSONDecodeError as exc:
        raise ValueError(f"Model did not return valid JSON: {content[:200]!r}") from exc

    missing = [key for key in schema.get("required", []) if key not in answer]

    if missing:
        raise ValueError(f"Model answer is missing keys {missing}: {answer}")

    stats = {"seconds": round(time.perf_counter() - started, 2),
             "prompt_tokens": response["prompt_eval_count"],
             "output_tokens": response["eval_count"]}

    return answer, stats


# ---- notebook 02: statistics SQLite lacks -------------------

def json_numbers(json_array):
    try:
        values = json.loads(json_array) if json_array else []
    except (TypeError, ValueError):
        return []

    numbers = []

    for value in values if isinstance(values, list) else []:
        if isinstance(value, (int, float)) and not isinstance(value, bool) \
                and not math.isnan(value):
            numbers.append(float(value))

    return numbers


def median_of(json_array):
    numbers = json_numbers(json_array)
    return statistics.median(numbers) if numbers else None


def stdev_of(json_array):
    numbers = json_numbers(json_array)
    return statistics.stdev(numbers) if len(numbers) > 1 else None


def variance_of(json_array):
    numbers = json_numbers(json_array)
    return statistics.variance(numbers) if len(numbers) > 1 else None


def percentile_of(json_array, percent):
    numbers = sorted(json_numbers(json_array))

    if not numbers or percent is None or not 0 <= percent <= 100:
        return None

    position = (len(numbers) - 1) * percent / 100
    lower, upper = math.floor(position), math.ceil(position)

    return numbers[lower] + (numbers[upper] - numbers[lower]) * (position - lower)


SQL_FUNCTIONS = {"median_of": (median_of, 1), "stdev_of": (stdev_of, 1),
                 "variance_of": (variance_of, 1),
                 "percentile_of": (percentile_of, 2)}


def register_functions(connection):
    for name, (function, arg_count) in SQL_FUNCTIONS.items():
        connection.create_function(name, arg_count, function,
                                   deterministic=True)
    return connection


# ---- running a query (full guardrails come in notebook 05) --

def run_query(database_path, sql):
    """Run SQL on a read-only connection and return a DataFrame."""

    uri = Path(database_path).resolve().as_uri() + "?mode=ro"

    with sqlite3.connect(uri, uri=True) as connection:
        register_functions(connection)
        return pd.read_sql_query(sql, connection)


def load_schema_text(dataset_name):
    return (DATASETS_DIR / dataset_name / "schema.txt").read_text(encoding="utf-8")


def database_path(dataset_name):
    return DATASETS_DIR / dataset_name / "dataset.db"


print(run_query(database_path("messy_sales"),
                "SELECT COUNT(*) AS orders FROM messy_sales"))

   orders
0      32


## 3. The prompt

Three parts:

1. **System message:** the rules every query must follow (SQLite only,
   one SELECT, exact category spelling, ISO dates, 1/0 booleans,
   readable column aliases, when to refuse).
2. **Few-shot examples:** question / answer pairs on a small made-up
   `orders` table, so the model learns the output style without copying
   real column names.
3. **The real request:** the dataset's schema text and the question.

In [3]:
# ============================================================
# 3. THE PROMPT
# ============================================================

SYSTEM_PROMPT = """You are a data analyst who answers questions by writing one SQLite query.

Rules:
1. Write exactly one read-only query that starts with SELECT or WITH. If the question asks to change, delete or add data, set can_answer to false and say in plan that only reading data is possible.
2. Use only the tables and columns in the schema. Never invent columns.
3. Text comparisons are case-sensitive: copy category values exactly as listed in the schema.
4. Dates are TEXT 'YYYY-MM-DD'. Compare them as text, group them with strftime('%Y-%m', col) or strftime('%Y', col).
   "in 2023" means col >= '2023-01-01' AND col < '2024-01-01'; "after 2023" means col >= '2024-01-01'; "before 2023" means col < '2023-01-01'.
5. Boolean columns store 1 for true and 0 for false.
6. Give every computed column a short snake_case alias, e.g. SUM(total_sales) AS total_sales.
7. Group only by the columns the question asks about, and select them so the result is readable. When two columns have similar names (rep, rep_2), use the one without a number unless the question names the other.
8. For "top N", "most" or "highest" use ORDER BY ... DESC LIMIT N (LIMIT 1 for a single winner).
9. Use ROUND(value, 2) for averages and ratios.
10. If the question needs a value that no column holds (for example cost, profit, margin, rating, age), set can_answer to false, say which column is missing in plan, and leave sql empty. Never invent a formula from other columns to stand in for it.

Fill "plan" first: one or two sentences naming the table, columns, filters and grouping you will use."""


SQL_ANSWER_SCHEMA = {
    "type": "object",
    "properties": {
        "plan": {"type": "string"},
        "can_answer": {"type": "boolean"},
        "sql": {"type": "string"},
    },
    "required": ["plan", "can_answer", "sql"],
}


EXAMPLE_SCHEMA = """TABLE orders (500 rows)
  order_id TEXT | unique per row | e.g. 'A001', 'A002', 'A003'
  order_date TEXT | date 'YYYY-MM-DD', 2023-01-01 to 2023-12-31
  store TEXT | values: 'Central', 'East', 'West'
  category TEXT | values: 'Books', 'Games', 'Toys'
  quantity INTEGER | range 1 to 20 | 4 missing
  amount REAL | currency amount | range 5.5 to 980.0
  returned INTEGER | boolean stored as 1 = true, 0 = false"""


FEW_SHOT_EXAMPLES = [
    ("How many orders came from the East store?",
     {"plan": "Count rows of orders where store is 'East'.",
      "can_answer": True,
      "sql": "SELECT COUNT(*) AS orders FROM orders WHERE store = 'East'"}),

    ("What is the total amount per category, highest first?",
     {"plan": "Sum amount from orders grouped by category, sorted descending.",
      "can_answer": True,
      "sql": "SELECT category, SUM(amount) AS total_amount FROM orders "
             "GROUP BY category ORDER BY total_amount DESC"}),

    ("Show the monthly number of returned orders.",
     {"plan": "Filter orders to returned = 1, group by month of order_date and count.",
      "can_answer": True,
      "sql": "SELECT strftime('%Y-%m', order_date) AS month, COUNT(*) AS returned_orders "
             "FROM orders WHERE returned = 1 GROUP BY month ORDER BY month"}),

    ("Which store has the highest median order amount?",
     {"plan": "Median of amount per store with median_of(json_group_array(amount)), keep the top one.",
      "can_answer": True,
      "sql": "SELECT store, median_of(json_group_array(amount)) AS median_amount "
             "FROM orders GROUP BY store ORDER BY median_amount DESC LIMIT 1"}),

    ("What was the average quantity of Toys orders in March 2023?",
     {"plan": "Average quantity of orders with category 'Toys' and order_date in 2023-03.",
      "can_answer": True,
      "sql": "SELECT ROUND(AVG(quantity), 2) AS average_quantity FROM orders "
             "WHERE category = 'Toys' AND order_date >= '2023-03-01' "
             "AND order_date < '2023-04-01'"}),

    ("How many orders were placed after 2022?",
     {"plan": "After 2022 means from 2023-01-01 onwards (2022 itself excluded), so count orders with order_date >= '2023-01-01'.",
      "can_answer": True,
      "sql": "SELECT COUNT(*) AS orders FROM orders WHERE order_date >= '2023-01-01'"}),

    ("Which customer spent the most?",
     {"plan": "The orders table has no customer column, so this cannot be answered.",
      "can_answer": False,
      "sql": ""}),

    ("What is the profit for each category?",
     {"plan": "Profit needs a cost column and orders has none; amount alone is revenue, so this cannot be answered.",
      "can_answer": False,
      "sql": ""}),
]


def user_message(schema_text, question):
    return f"Schema:\n{schema_text}\n\nQuestion: {question}"


def build_messages(schema_text, question):
    """System rules + few-shot pairs + the real schema and question."""

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    for example_question, example_answer in FEW_SHOT_EXAMPLES:
        messages.append({"role": "user",
                         "content": user_message(EXAMPLE_SCHEMA, example_question)})
        messages.append({"role": "assistant",
                         "content": json.dumps(example_answer)})

    messages.append({"role": "user",
                     "content": user_message(schema_text, question)})

    return messages


preview = build_messages(load_schema_text("messy_sales"), "How many orders are there?")
print(f"{len(preview)} messages, ~{sum(len(m['content']) for m in preview) // 4} tokens")
print(preview[-1]["content"][:600])

18 messages, ~2106 tokens
Schema:
TABLE messy_sales (32 rows)
  order_id TEXT | unique per row | e.g. '00101', '00102', '00103'
  order_date TEXT | date 'YYYY-MM-DD', 2024-01-02 to 2024-03-30
  customer_name TEXT | values: 'Amit Shah', 'Arjun Mehta', 'Asha Verma', 'Kavya Iyer', 'Neha Gupta', 'Priya Nair', 'Rahul Singh', 'Rohan Das', 'Sneha Kulkarni', 'Vikram Rao'
  city TEXT | values: 'Bengaluru', 'Delhi', 'Mumbai', 'Mumbay', 'New Delhi', 'Pune'
  region TEXT | values: 'North', 'South', 'West'
  product TEXT | values: 'Headphones', 'Keyboard', 'Laptop', 'Monitor', 'Mouse'
  units_sold INTEGER | range 1 to 10 | 3 missin


## 4. Generating SQL

In [4]:
# ============================================================
# 4. GENERATING SQL
# ============================================================

def clean_sql(sql):
    """Strip whitespace, code fences and a trailing semicolon."""

    sql = sql.strip()

    if sql.startswith("```"):
        sql = sql.strip("`")
        sql = sql[3:] if sql.lower().startswith("sql") else sql

    return sql.strip().rstrip(";").strip()


def generate_sql(settings, schema_text, question):
    """
    Ask the model for a query.

    Returns:
        dict with plan, can_answer, sql, seconds, prompt_tokens
    """

    messages = build_messages(schema_text, question)
    answer, stats = chat_json(settings, messages, SQL_ANSWER_SCHEMA)

    return {
        "plan": answer["plan"].strip(),
        "can_answer": bool(answer["can_answer"]),
        "sql": clean_sql(answer["sql"]),
        **stats,
    }


result = generate_sql(model_settings, load_schema_text("messy_sales"),
                      "Which city had the highest total sales?")
print(json.dumps(result, indent=2))
run_query(database_path("messy_sales"), result["sql"])

{
  "plan": "Sum total_sales for each city and find the one with the maximum sum.",
  "can_answer": true,
  "sql": "SELECT city, SUM(total_sales) AS total_sales_sum FROM messy_sales GROUP BY city ORDER BY total_sales_sum DESC LIMIT 1",
  "seconds": 4.02,
  "prompt_tokens": 2829,
  "output_tokens": 57
}


,city,total_sales_sum
0,Delhi,11275024


## 5. Comparing results

A generated query is correct when its result holds the same values as
the hand-written query: same number of rows, and every expected column
found among the generated columns (names, column order and row order
ignored, numbers equal within rounding). Extra columns are fine.

In [5]:
# ============================================================
# 5. COMPARING RESULTS
# ============================================================

def normalize_value(value):
    """Numbers -> float, missing -> None, everything else -> text."""

    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None

    if isinstance(value, bool):
        return float(value)

    if isinstance(value, (int, float)):
        return float(value)

    if hasattr(value, "item"):                      # numpy scalars
        return normalize_value(value.item())

    return str(value)


def sort_key(value):
    """Numbers first, then text, then missing values."""

    if isinstance(value, float):
        return (0, value, "")
    if isinstance(value, str):
        return (1, 0.0, value)
    return (2, 0.0, "")


def sorted_column(series):
    return sorted((normalize_value(value) for value in series), key=sort_key)


def values_equal(first, second):
    if isinstance(first, float) and isinstance(second, float):
        return math.isclose(first, second, rel_tol=RELATIVE_TOLERANCE,
                            abs_tol=0.01)
    return first == second


def columns_equal(first, second):
    return len(first) == len(second) and all(
        values_equal(a, b) for a, b in zip(first, second))


def results_match(generated, expected):
    """True when `generated` contains every column of `expected`."""

    if len(generated) != len(expected):
        return False

    available = [sorted_column(generated.iloc[:, position])
                 for position in range(generated.shape[1])]

    for position in range(expected.shape[1]):
        wanted = sorted_column(expected.iloc[:, position])
        match = next((index for index, column in enumerate(available)
                      if columns_equal(column, wanted)), None)

        if match is None:
            return False

        available.pop(match)

    return True


# Unit checks for the comparison itself.
a = pd.DataFrame({"city": ["Pune", "Delhi"], "total": [10, 20.004]})
b = pd.DataFrame({"t": [20.0, 10.0]})
assert results_match(a, b)                               # extra column, order
assert not results_match(a, pd.DataFrame({"t": [20.0, 11.0]}))
assert not results_match(a.head(1), b)                   # row count
assert results_match(pd.DataFrame({"x": [None, 1]}), pd.DataFrame({"y": [1.0, None]}))
print("Comparison checks passed.")

Comparison checks passed.


## 6. Test questions

Hand-written correct queries over the sales file and the two-sheet
Excel file, kept in `data/input_data/dev_questions.json` (notebook 05
uses the same file). A `null` query means the model should refuse.

In [6]:
# ============================================================
# 6. TEST QUESTIONS
# ============================================================

DEV_QUESTIONS_PATH = BASE_DIR / "data" / "input_data" / "dev_questions.json"


def load_questions(path):
    """[{"dataset", "question", "expected_sql"}] -> list of tuples."""

    with path.open(encoding="utf-8") as file:
        return [(item["dataset"], item["question"], item["expected_sql"])
                for item in json.load(file)]


TEST_QUESTIONS = load_questions(DEV_QUESTIONS_PATH)

print(len(TEST_QUESTIONS), "questions")

16 questions


## 7. First-attempt accuracy

Every question once, no retries. A question counts as correct when the
query runs and its result matches, or when the model correctly refuses.
The same questions also go to the 3B model (run on the GPU here; speed
differs on a CPU, accuracy does not).

In [7]:
# ============================================================
# 7. FIRST-ATTEMPT ACCURACY
# ============================================================

def score_question(settings, dataset_name, question, expected_sql):
    """Generate, run and score one question."""

    row = {"dataset": dataset_name, "question": question,
           "model": settings["model"]}

    # Step 1: generate.
    try:
        generated = generate_sql(settings, load_schema_text(dataset_name), question)

    except ValueError as exc:
        return {**row, "status": "bad_json", "correct": False, "error": str(exc)}

    row.update({"plan": generated["plan"], "sql": generated["sql"],
                "can_answer": generated["can_answer"],
                "seconds": generated["seconds"]})

    # Step 2: questions the data cannot answer must be refused.
    if expected_sql is None:
        refused = not generated["can_answer"]
        return {**row, "status": "refused" if refused else "answered",
                "correct": refused}

    if not generated["can_answer"] or not generated["sql"]:
        return {**row, "status": "refused", "correct": False}

    # Step 3: run both queries and compare.
    try:
        result = run_query(database_path(dataset_name), generated["sql"])

    except Exception as exc:
        return {**row, "status": "sql_error", "correct": False,
                "error": f"{type(exc).__name__}: {exc}"}

    expected = run_query(database_path(dataset_name), expected_sql)
    correct = results_match(result, expected)

    return {**row, "status": "ok" if correct else "wrong_result",
            "correct": correct, "rows": len(result)}


def run_test_set(settings, questions=TEST_QUESTIONS):
    rows = []

    for dataset_name, question, expected_sql in questions:
        rows.append(score_question(settings, dataset_name, question, expected_sql))
        logger.info("%-13s %-55s %s", rows[-1]["status"], question[:55],
                    settings["model"])

    return pd.DataFrame(rows)


small_settings = {**model_settings, "model": "qwen2.5:3b-instruct",
                  "num_ctx": 4096}

results = pd.concat([run_test_set(model_settings),
                     run_test_set(small_settings)], ignore_index=True)

summary = results.groupby("model").agg(
    correct=("correct", "sum"), questions=("correct", "size"),
    avg_seconds=("seconds", "mean"))
summary["accuracy"] = (summary["correct"] / summary["questions"]).round(2)
summary

2026-11-14 18:12:40,638 | INFO | ok            How many orders are there?                              qwen2.5:7b-instruct


2026-11-14 18:12:43,825 | INFO | ok            How many orders were placed in Delhi?                   qwen2.5:7b-instruct


2026-11-14 18:12:47,002 | INFO | ok            What are the total sales by region?                     qwen2.5:7b-instruct


2026-11-14 18:12:50,232 | INFO | ok            Which product sold the most units?                      qwen2.5:7b-instruct


2026-11-14 18:12:53,563 | INFO | ok            Show total sales for each month.                        qwen2.5:7b-instruct


2026-11-14 18:12:56,807 | INFO | ok            What is the average discount on paid orders?            qwen2.5:7b-instruct


... (23 outputs left out)

2026-11-14 18:14:21,698 | INFO | ok            How many active employees work in Sales?                qwen2.5:3b-instruct


2026-11-14 18:14:24,838 | INFO | ok            How many employees joined after 2020?                   qwen2.5:3b-instruct


2026-11-14 18:14:27,791 | INFO | ok            What is the Q2 target for the Finance department?       qwen2.5:3b-instruct


,correct,questions,avg_seconds,accuracy
model,,,,
qwen2.5:3b-instruct,16,16,3.392500,1.00
qwen2.5:7b-instruct,15,16,3.055625,0.94


In [8]:
pd.set_option("display.max_colwidth", 120)
failures = results.loc[~results["correct"],
                       ["model", "question", "status", "sql", "error"]
                       if "error" in results else
                       ["model", "question", "status", "sql"]]
failures

,model,question,status,sql
14,qwen2.5:7b-instruct,How many employees joined after 2020?,wrong_result,SELECT COUNT(*) AS employees FROM messy_employees_employees WHERE joining_date >= '2020-01-01'


## 8. Checks

Deterministic parts only; accuracy is measured above, not asserted.

In [9]:
# ============================================================
# 8. CHECKS
# ============================================================

# Prompt: rules, all few-shot pairs, real schema last.
messages = build_messages(load_schema_text("messy_sales"), "test question")
assert messages[0]["role"] == "system" and "case-sensitive" in messages[0]["content"]
assert len(messages) == 1 + 2 * len(FEW_SHOT_EXAMPLES) + 1
assert "TABLE messy_sales" in messages[-1]["content"]
assert messages[-1]["content"].endswith("Question: test question")

# Few-shot answers follow the output schema.
for _, example in FEW_SHOT_EXAMPLES:
    assert set(example) == set(SQL_ANSWER_SCHEMA["required"])

# Every hand-written query runs.
for dataset_name, _, expected_sql in TEST_QUESTIONS:
    if expected_sql:
        run_query(database_path(dataset_name), expected_sql)

# SQL clean-up.
assert clean_sql("```sql\nSELECT 1;\n```") == "SELECT 1"
assert clean_sql("  SELECT 1 ; ") == "SELECT 1"

# Read-only connection: writes are refused.
try:
    run_query(database_path("messy_sales"), "DELETE FROM messy_sales")
    raise AssertionError("write was allowed")
except Exception as exc:
    assert "readonly" in str(exc).lower() or "read-only" in str(exc).lower(), exc

# Every model answer was valid JSON with the expected keys.
assert (results["status"] != "bad_json").all()

print("All checks passed.")

All checks passed.


## 9. Save outputs

In [10]:
# ============================================================
# 9. SAVE OUTPUTS
# ============================================================

results.to_csv(OUTPUT_DIR / "first_attempt_results.csv", index=False)
summary.to_csv(OUTPUT_DIR / "first_attempt_summary.csv")

with (OUTPUT_DIR / "prompt_example.json").open("w", encoding="utf-8") as file:
    json.dump(build_messages(load_schema_text("messy_sales"),
                             "How many orders are there?"),
              file, indent=2, ensure_ascii=False)

print(sorted(path.name for path in OUTPUT_DIR.iterdir()))

['first_attempt_results.csv', 'first_attempt_summary.csv', 'prompt_example.json']
